<a href="https://colab.research.google.com/github/ipeirotis-org/datasets/blob/main/NBA/nba_finals_leadtime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Finals Lead-Time Analysis (1997 to 2026)

This notebook reproduces, from scratch, a single statistic for every NBA Finals since
the league began publishing play-by-play data:

> **For each Finals series, what fraction of elapsed game time was the eventual champion
> ahead on the scoreboard?**

It pulls play-by-play for every Finals game, reconstructs the running score as a function
of the game clock, and integrates how long the champion was leading, trailing, and tied
across the whole series (games the champion lost are included, since the question is about
the eventual winner across the series).

**Pipeline**
1. Identify the Finals games for each season.
2. Pull play-by-play for each game.
3. Convert each event to elapsed seconds and reconstruct the running margin.
4. Integrate lead / trail / tie time per game.
5. Aggregate to one row per series.

**Data source:** `nba_api` (the public stats.nba.com endpoints). Play-by-play with scores
is available from the 1996-97 season onward, so the earliest Finals we can analyze is 1997.


## 1. Setup

`nba_api` is a thin client over the stats.nba.com endpoints. No API key is required, but the
endpoints can be slow and occasionally time out, so the driver loop below caches results to
disk and is safe to re-run.


In [3]:
!pip install nba_api

In [4]:
# !pip install nba_api pandas
import re, time, os, warnings
import pandas as pd
from nba_api.stats.endpoints import leaguegamefinder, playbyplayv3

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

## 2. Configuration

`SEASONS` is the list of season strings to process. 1996-97 is the first season with
play-by-play, and we run through the current season. Per-game results are cached so that a
re-run only fetches games it has not seen before.


In [5]:
SEASONS = [f"{y}-{str(y+1)[-2:]}" for y in range(1996, 2026)]  # 1996-97 .. 2025-26
CACHE   = "series_games.csv"   # per-game results
print(len(SEASONS), "seasons:", SEASONS[0], "...", SEASONS[-1])

30 seasons: 1996-97 ... 2025-26


## 3. Game-clock helpers

The play-by-play clock is reported as time **remaining in the current period**, formatted as
an ISO 8601 duration, for example `PT07M36.00S` (7 minutes 36 seconds left). To place every
event on a single timeline we convert it to **elapsed seconds since tipoff**.

Period lengths:
* Regulation periods 1 to 4 are 12 minutes (720 seconds).
* Each overtime period is 5 minutes (300 seconds).


In [6]:
def remaining_seconds(clock: str):
    '''Parse the play-by-play clock (time left in the period) into seconds.'''
    m = re.match(r"PT(\d+)M([\d.]+)S", clock)
    return int(m.group(1)) * 60 + float(m.group(2)) if m else None

def period_length(period: int) -> float:
    return 720.0 if period <= 4 else 300.0

def period_start_elapsed(period: int) -> float:
    '''Elapsed seconds at the start of a given period.'''
    if period <= 4:
        return (period - 1) * 720.0
    return 4 * 720.0 + (period - 5) * 300.0

def elapsed_seconds(period: int, clock: str):
    '''Elapsed seconds since tipoff for an event in `period` with `clock` remaining.'''
    rem = remaining_seconds(clock)
    if rem is None:
        return None
    return period_start_elapsed(period) + (period_length(period) - rem)

## 4. Identifying the Finals games

There is no direct "Finals only" filter, and the game-ID scheme is not stable across eras:

* Older seasons use sequential playoff IDs (`0049600083`).
* Recent seasons encode the round (`0042500401`, where `004` is the round = Finals).

A method that works for every season relies on the bracket itself: the two Finals teams come
from different conferences, so the **only** playoff games between them are the Finals. We take
the two teams in the last playoff game of the season (the finalists), then collect every
playoff game in which both of those teams appear. The champion is the winner of the final game.


In [7]:
def get_finals(season: str):
    '''Return per-game dicts, champion abbr, and opponent abbr for a Finals series.'''
    gf = leaguegamefinder.LeagueGameFinder(
        season_nullable=season, season_type_nullable="Playoffs",
        league_id_nullable="00", timeout=45)
    df = gf.get_data_frames()[0].sort_values("GAME_DATE")

    # Finalists = the two teams in the last playoff game of the season.
    last_gid   = df.drop_duplicates("GAME_ID").iloc[-1].GAME_ID
    finalists  = set(df[df.GAME_ID == last_gid].TEAM_ABBREVIATION)

    # Finals games = playoff games where BOTH finalists appear.
    g       = df[df.TEAM_ABBREVIATION.isin(finalists)]
    counts  = g.groupby("GAME_ID").size()
    fin_ids = [gid for gid, c in counts.items() if c == 2]
    sub     = g[g.GAME_ID.isin(fin_ids)]

    # Champion = winner of the chronologically last Finals game.
    last_fin = sub.sort_values("GAME_DATE").GAME_ID.iloc[-1]
    champ    = sub[(sub.GAME_ID == last_fin) & (sub.WL == "W")].iloc[0].TEAM_ABBREVIATION
    opp      = [t for t in finalists if t != champ][0]

    games = []
    for gid in sorted(fin_ids):
        crow = sub[(sub.GAME_ID == gid) & (sub.TEAM_ABBREVIATION == champ)].iloc[0]
        games.append({
            "gid": gid, "season": season, "champ": champ, "opp": opp,
            "champ_is_home": "vs." in crow.MATCHUP,   # MATCHUP uses 'vs.' for home, '@' for away
            "champ_won": crow.WL == "W",
            "date": crow.GAME_DATE,
        })
    return games, champ, opp

# Quick check on one season:
_g, _c, _o = get_finals("2024-25")
print(_c, "def.", _o, "  games:", len(_g))

OKC def. IND   games: 7


## 5. Reconstructing the running score (and a data quirk worth knowing)

`PlayByPlayV3` returns `scoreHome` / `scoreAway` columns that hold the **running** score. Only
scoring events change the score, so we forward-fill across non-scoring rows.

There is one trap. In the **older** play-by-play format, non-scoring rows do not leave the score
blank: they carry a literal `0`. For example, the row for a rebound mid-game reads `scoreHome=0,
scoreAway=0` rather than blank. A naive forward-fill then resets the score to 0-0 on roughly half
the rows, which silently turns large stretches of the game into spurious "ties."

The fix: treat a row where **both** scores are zero as a non-update (set it to missing, then
forward-fill, then fill any leading gap with 0). A real 0-0 only occurs before the first basket,
so this is safe, and it leaves the modern format untouched.


In [8]:
def reconstruct_scores(df: pd.DataFrame):
    '''Return (home_score, away_score) Series as step functions over the rows.'''
    sH = pd.to_numeric(df["scoreHome"], errors="coerce")
    sA = pd.to_numeric(df["scoreAway"], errors="coerce")
    both_zero = (sH == 0) & (sA == 0)          # non-scoring rows in older PBP carry literal 0-0
    sH = sH.mask(both_zero)
    sA = sA.mask(both_zero)
    return sH.ffill().fillna(0), sA.ffill().fillna(0)

## 6. Per-game lead-time

For one game we build a step function of the champion's margin over elapsed time and integrate:

* The margin established at event *i* holds until event *i+1*, so the time on each interval is
  classified by the sign of the margin at its start.
* From tipoff to the first basket the margin is 0 (tied); after the final basket it holds to the
  buzzer.

We return total game seconds `T` and the seconds the champion spent leading, trailing, and tied,
plus the final score from the champion's perspective.


In [9]:
def analyze_game(gid: str, champ_is_home: bool):
    df = playbyplayv3.PlayByPlayV3(game_id=gid, timeout=45).get_data_frames()[0]
    df["sH"], df["sA"] = reconstruct_scores(df)
    df["elapsed"] = [elapsed_seconds(p, c) for p, c in zip(df.period, df.clock)]
    df = df.dropna(subset=["elapsed"]).sort_values("elapsed").reset_index(drop=True)

    # Champion margin = champ score minus opponent score.
    df["margin"] = (df.sH - df.sA) if champ_is_home else (df.sA - df.sH)

    T = df["elapsed"].max()
    lead = trail = tie = 0.0
    el, m = df["elapsed"].values, df["margin"].values
    for i in range(len(df) - 1):
        dur = el[i + 1] - el[i]
        if dur <= 0:
            continue
        if   m[i] > 0: lead  += dur
        elif m[i] < 0: trail += dur
        else:          tie   += dur

    fh, fa    = int(df.sH.iloc[-1]), int(df.sA.iloc[-1])
    champ_pts = fh if champ_is_home else fa
    opp_pts   = fa if champ_is_home else fh
    return {"T": T, "lead": lead, "trail": trail, "tie": tie,
            "champ_pts": champ_pts, "opp_pts": opp_pts}

## 7. Driver loop

Iterate the seasons, fetch each Finals game, and append results to the cache. The loop skips
games already cached, so it can be interrupted and re-run. A small sleep is polite to the
endpoint and reduces timeouts.


In [10]:
def build_cache(seasons=SEASONS, cache=CACHE, pause=0.4):
    done = set(pd.read_csv(cache, dtype={"gid": str}).gid) if os.path.exists(cache) else set()
    for season in seasons:
        try:
            games, champ, opp = get_finals(season)
        except Exception as e:
            print("finder fail", season, repr(e)[:80]); continue
        for info in games:
            if info["gid"] in done:
                continue
            try:
                res = analyze_game(info["gid"], info["champ_is_home"])
                row = {**info, **res}
                pd.DataFrame([row]).to_csv(cache, mode="a",
                    header=not os.path.exists(cache), index=False)
                done.add(info["gid"])
            except Exception as e:
                print("game fail", info["gid"], repr(e)[:80])
            time.sleep(pause)
        print(f"done {season} ({len(games)} games)")
    print("ALL DONE")

build_cache()

done 1996-97 (6 games)
done 1997-98 (6 games)
done 1998-99 (5 games)
done 1999-00 (6 games)
done 2000-01 (5 games)
done 2001-02 (4 games)
done 2002-03 (6 games)
done 2003-04 (5 games)
done 2004-05 (7 games)
done 2005-06 (6 games)
done 2006-07 (4 games)
done 2007-08 (6 games)
done 2008-09 (5 games)
done 2009-10 (7 games)
done 2010-11 (6 games)
done 2011-12 (5 games)
done 2012-13 (7 games)
done 2013-14 (5 games)
done 2014-15 (6 games)
done 2015-16 (7 games)
done 2016-17 (5 games)
done 2017-18 (4 games)
done 2018-19 (6 games)
done 2019-20 (6 games)
done 2020-21 (6 games)
done 2021-22 (6 games)
done 2022-23 (5 games)
done 2023-24 (5 games)
done 2024-25 (7 games)
done 2025-26 (5 games)
ALL DONE


## 8. Aggregate to one row per series

Pool seconds across every game of a series: the series lead share is total champion-leading
seconds divided by total elapsed seconds. The series score is the champion's wins versus losses
among Finals games.


In [11]:
games = pd.read_csv(CACHE, dtype={"gid": str})

rows = []
for season, g in games.groupby("season"):
    champ_wins = int(g.champ_won.sum())
    opp_wins   = len(g) - champ_wins
    total      = g["T"].sum()
    rows.append({
        "year":   int(season[:4]) + 1,
        "champ":  g.champ.iloc[0],
        "opp":    g.opp.iloc[0],
        "score":  f"{champ_wins}-{opp_wins}",
        "games":  len(g),
        "led_pct":    100 * g["lead"].sum()  / total,
        "tied_pct":   100 * g["tie"].sum()   / total,
        "trailed_pct":100 * g["trail"].sum() / total,
    })

series = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
series.round(1)

,year,champ,opp,score,games,led_pct,tied_pct,trailed_pct
0,1997,CHI,UTA,4-2,6,30.3,10.4,59.3
1,1998,CHI,UTA,4-2,6,48.1,8.4,43.4
2,1999,SAS,NYK,4-1,5,56.0,6.2,37.7
3,2000,LAL,IND,4-2,6,36.8,6.1,57.1
4,2001,LAL,PHI,4-1,5,66.8,8.7,24.5
5,2002,LAL,NJN,4-0,4,82.3,5.3,12.4
6,2003,SAS,NJN,4-2,6,43.5,7.0,49.5
7,2004,DET,LAL,4-1,5,65.3,10.7,23.9
8,2005,SAS,DET,4-3,7,40.2,11.0,48.8
9,2006,MIA,DAL,4-2,6,45.2,6.9,47.9


## 9. Results and summary

In [12]:
view = series.copy()
view["matchup"] = view.champ + " d. " + view.opp
print(view[["year", "matchup", "score", "led_pct", "trailed_pct"]]
      .to_string(index=False, formatters={"led_pct": "{:.1f}".format,
                                          "trailed_pct": "{:.1f}".format}))

print()
print(f"Seasons analyzed     : {len(series)}  ({series.year.min()}-{series.year.max()})")
print(f"Mean champion led    : {series.led_pct.mean():.1f}%")
print(f"Median champion led  : {series.led_pct.median():.1f}%")
top = series.loc[series.led_pct.idxmax()]
bot = series.loc[series.led_pct.idxmin()]
print(f"Most dominant        : {int(top.year)} {top.champ} ({top.led_pct:.1f}%)")
print(f"Least dominant       : {int(bot.year)} {bot.champ} ({bot.led_pct:.1f}%)")

 year    matchup score led_pct trailed_pct
 1997 CHI d. UTA   4-2    30.3        59.3
 1998 CHI d. UTA   4-2    48.1        43.4
 1999 SAS d. NYK   4-1    56.0        37.7
 2000 LAL d. IND   4-2    36.8        57.1
 2001 LAL d. PHI   4-1    66.8        24.5
 2002 LAL d. NJN   4-0    82.3        12.4
 2003 SAS d. NJN   4-2    43.5        49.5
 2004 DET d. LAL   4-1    65.3        23.9
 2005 SAS d. DET   4-3    40.2        48.8
 2006 MIA d. DAL   4-2    45.2        47.9
 2007 SAS d. CLE   4-0    76.6        18.4
 2008 BOS d. LAL   4-2    42.8        50.1
 2009 LAL d. ORL   4-1    51.5        37.7
 2010 LAL d. BOS   4-3    52.6        39.7
 2011 DAL d. MIA   4-2    42.6        47.7
 2012 MIA d. OKC   4-1    73.3        20.7
 2013 MIA d. SAS   4-3    41.6        49.1
 2014 SAS d. MIA   4-1    74.5        19.5
 2015 GSW d. CLE   4-2    45.0        46.3
 2016 CLE d. GSW   4-3    49.5        42.6
 2017 GSW d. CLE   4-1    58.5        38.0
 2018 GSW d. CLE   4-0    70.2        24.8
 2019 TOR d

## 10. Optional: drill into a single year

`game_table(year)` shows the per-game breakdown behind any series, from the champion's
perspective (so a game the champion lost shows mostly trailing time).


In [13]:
def game_table(year):
    season = f"{year-1}-{str(year)[-2:]}"
    g = games[games.season == season].sort_values("gid").reset_index(drop=True)
    out = pd.DataFrame({
        "game":     range(1, len(g) + 1),
        "result":   ["W" if w else "L" for w in g.champ_won],
        "champ":    g.champ_pts,
        "opp":      g.opp_pts,
        "OT":       ["OT" if t > 2880 else "" for t in g["T"]],
        "led_pct":  (100 * g["lead"]  / g["T"]).round(1),
        "trailed_pct": (100 * g["trail"] / g["T"]).round(1),
    })
    return out

game_table(2026)   # the 2026 Knicks: champions who mostly trailed

,game,result,champ,opp,OT,led_pct,trailed_pct
0,1,W,105,95,,40.7,55.0
1,2,W,105,104,,49.4,47.0
2,3,L,111,115,,16.0,77.8
3,4,W,107,106,,1.9,97.1
4,5,W,94,90,,9.5,84.2


## Methodology notes and caveats

* **What "leading" means.** Time is classified strictly: leading (margin > 0), trailing
  (margin < 0), or tied (margin = 0). Tie time is reported separately rather than split, so
  led% + trailed% + tied% = 100%.
* **Series-level, champion perspective.** Each series figure pools all of its games, including
  games the champion lost. This answers "how often was the eventual winner ahead during the
  series." A different and higher number (around 66% over 2015-2025) comes from measuring the
  **per-game** winner, that is, the team that won each individual game. The two questions are
  not the same.
* **Pooled vs per-game mean.** The series figure pools seconds (sum of leading seconds over sum
  of total seconds). Averaging each game's percentage first gives a very similar result because
  Finals games are close in length, apart from overtime.
* **Overtime** is handled directly: OT periods add 300 seconds each, so `T` exceeds 2880 for OT
  games and the denominator grows accordingly.
* **Data coverage.** Play-by-play with scores starts in 1996-97. Earlier Finals cannot be
  reconstructed this way from this source.
* **Endpoint reliability.** stats.nba.com can rate-limit or time out. The cache makes the driver
  resumable; raising the per-call `timeout` or adding retries helps on flaky connections.
